## Dataset

This exercise uses the UCI Wine Quality dataset. Download `winequality-red.csv` from the official UCI source and place it at `data/winequality-red.csv` before running the notebook.

Source: https://archive.ics.uci.edu/dataset/186/wine+quality


# Wine Quality — Machine Learning Regression Pipeline

## Objective

Build a complete **regression pipeline from scratch** using the Wine Quality dataset.

The purpose of this notebook is to understand the workflow:

**Dataset → Inspection → Cleaning → EDA → Features/Target → Train/Test Split → Model → Prediction → Evaluation → Experiments**

This is a learning experiment, so we will focus on **why each step exists**, not just on getting a prediction.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Load the Dataset

The dataset contains numerical physicochemical measurements of red wine and a `quality` score.

For this first experiment, we keep the problem simple:

- All input features are numerical.
- `quality` is the target.
- We will initially treat `quality` as a **regression target** because it is represented as a numerical value.


In [ ]:
df = pd.read_csv("data/winequality-red.csv", sep=";")


## 2. Initial Data Inspection

We start with a small set of standard checks:

- `head()` → inspect the first few rows.
- `shape` → understand the number of rows and columns.
- `info()` → inspect data types and non-null counts.
- `describe()` → inspect numerical distributions and ranges.

These checks help us understand the dataset before making modelling decisions.


In [ ]:
df.head()


In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Duplicate Records

Before training a model, we should check whether exact duplicate rows exist.

Why?

Duplicate observations can give repeated information to the model and may affect the evaluation if the same observation appears more than once.

We first **inspect** duplicates before deciding how to handle them.


In [ ]:
df.duplicated().sum()

In [ ]:
df[df.duplicated(keep=False)].sort_values(
    by=list(df.columns)
).head(20)

In [ ]:
df[df.duplicated()].head()

## 4. Target Exploration

Our target variable is `quality`.

We examine:

- frequency of each quality value,
- descriptive statistics,
- relative proportions.

This tells us how the target is distributed and whether some values are much more common than others.


In [ ]:
df["quality"].value_counts().sort_index()

In [ ]:
df["quality"].describe()

In [ ]:
df["quality"].value_counts(normalize=True).sort_index()

## 5. Data Cleaning

The dataset contains exact duplicate rows.

For this experiment, we remove exact duplicates:

```python
df = df.drop_duplicates()
```

After cleaning, we verify the resulting shape and confirm that no exact duplicates remain.


In [ ]:
df = df.drop_duplicates()
df.shape
df.duplicated().sum()

## 6. Basic Exploratory Data Analysis (EDA)

Exploratory Data Analysis (EDA) means examining the data to understand its structure, distributions, relationships, and possible problems before modelling.

Here we start with the target distribution.

We are **not trying to perform every possible EDA operation**. We choose EDA based on the modelling problem and what the data tells us.


In [ ]:
df["quality"].hist(bins=range(3, 10))
plt.xlabel("Quality")
plt.ylabel("Number of wines")
plt.title("Wine Quality Distribution")
plt.show()

## 7. Separate Features and Target

Machine learning models need us to distinguish between:

- **Features (X):** input variables used to make predictions.
- **Target (y):** the variable we want to predict.

Here:

- `X` = the 11 physicochemical measurements.
- `y` = `quality`.

The model will learn a relationship:

**X → quality**


In [ ]:
X = df.drop("quality", axis=1)
y = df["quality"]

print("X shape:", X.shape)
print("y shape:", y.shape)

## 8. Train-Test Split

We divide the data into:

- **Training set:** used to learn the model parameters.
- **Test set:** kept aside to evaluate performance on unseen data.

We use 80% for training and 20% for testing.

`random_state=42` makes the split reproducible, so we get the same split each time we run the notebook.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

## 9. Baseline Model — Linear Regression

We start with a simple baseline rather than immediately choosing a complex algorithm.

### Linear Regression

Linear Regression assumes that the target can be approximated by a linear combination of the input features:

**ŷ = β₀ + β₁X₁ + β₂X₂ + ... + βₙXₙ**

where:

- `ŷ` = predicted target
- `β₀` = intercept
- `β₁ ... βₙ` = learned coefficients
- `X₁ ... Xₙ` = input features

The model learns the coefficients from the training data.


In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train, y_train)

## 10. Generate Predictions

After training, we use the trained model to predict `quality` for observations in the test set.

Important distinction:

**Training:** learn the model.

**Prediction:** use the learned model on new/unseen feature values.


In [ ]:
y_pred = model.predict(X_test)

## 11. Model Evaluation

We evaluate the predictions using four common regression metrics.

### Mean Absolute Error (MAE)

**Mean Absolute Error** measures the average absolute difference between actual and predicted values.

**Lower is better.**

### Mean Squared Error (MSE)

**Mean Squared Error** averages the squared prediction errors, giving larger errors more weight.

**Lower is better.**

### Root Mean Squared Error (RMSE)

**Root Mean Squared Error** is the square root of Mean Squared Error.

It is expressed in the same units as the target.

**Lower is better.**

### R-squared (R²)

**R-squared**, also called the coefficient of determination, measures how much of the variation in the target is explained by the model relative to a mean-prediction baseline.

For the usual interpretation, **higher is better**.

There is no universal cutoff that makes an R² value "good" or "bad"; we compare it with baselines, alternative models, and the requirements of the problem.


In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R²  :", r2)

## 12. Actual vs Predicted Values

A simple comparison of actual and predicted values helps us understand individual predictions.

This is not a replacement for proper evaluation metrics, but it gives us a concrete view of model behaviour.


In [ ]:
comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

comparison.head(10)

## 13. Feature Scaling Experiment

Some machine learning algorithms are sensitive to the scale of input features.

**StandardScaler** standardizes each feature approximately to:

- mean = 0
- standard deviation = 1

Important rule:

**Fit the scaler only on the training data, then transform both training and test data.**

```python
scaler.fit_transform(X_train)
scaler.transform(X_test)
```

We are experimenting with scaling here to understand its effect. Scaling is not automatically required for every algorithm.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Train Linear Regression on Scaled Features

We train the same Linear Regression model using the scaled features.

Because Linear Regression is being used here, we can compare the scaled and unscaled results directly.


In [ ]:
model_scaled = LinearRegression()

model_scaled.fit(X_train_scaled, y_train)

y_pred_scaled = model_scaled.predict(X_test_scaled)

### Compare Scaled vs Unscaled Results

If the evaluation metrics remain essentially unchanged, that tells us that scaling did not materially affect this particular Linear Regression experiment.

This does **not** mean scaling is unnecessary in machine learning generally. Its importance depends on the algorithm and the data.


In [ ]:
mae_scaled = mean_absolute_error(y_test, y_pred_scaled)
mse_scaled = mean_squared_error(y_test, y_pred_scaled)
rmse_scaled = np.sqrt(mse_scaled)
r2_scaled = r2_score(y_test, y_pred_scaled)

print("Scaled Linear Regression")
print("MAE :", mae_scaled)
print("MSE :", mse_scaled)
print("RMSE:", rmse_scaled)
print("R²  :", r2_scaled)

## 14. Training Performance

We now generate predictions on the **training set**.

This allows us to compare how the model performs on data it learned from versus data it did not see during training.


In [ ]:
y_train_pred = model.predict(X_train)

## 15. Training vs Test Performance

Comparing training and test performance helps us reason about **generalization**.

### Typical patterns

| Training | Test | Possible interpretation |
|---|---|---|
| Poor | Poor | Possible underfitting |
| Good | Similar to training | Good generalization |
| Very good | Much worse | Possible overfitting |

There is no universal numerical cutoff for the size of the gap. We look at the pattern, the problem, the metric, and supporting evidence.


In [ ]:
from sklearn.metrics import r2_score

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_pred)

print("Training R²:", train_r2)
print("Test R²    :", test_r2)

## 16. Final Baseline Evaluation

We print the training and test values for:

- **Mean Absolute Error (MAE)**
- **Mean Squared Error (MSE)**
- **Root Mean Squared Error (RMSE)**
- **R-squared (R²)**

For this experiment, the training and test results are relatively close, so there is **no obvious large generalization gap** suggesting strong overfitting.

However, this comparison alone does not prove that Linear Regression is the best model or that it is not underfitting.

## Current Conclusion

We now have a complete first regression pipeline:

**Load → Inspect → Clean → Explore → Split → Train → Predict → Evaluate → Compare**

The next question is:

> **Can a more flexible model capture relationships that Linear Regression is missing?**

That leads naturally to the next experiment: **Polynomial Regression and model complexity.**


In [ ]:
train_mae = mean_absolute_error(y_train, y_train_pred)
train_mse = mean_squared_error(y_train, y_train_pred)
train_rmse = np.sqrt(train_mse)

print("Training Mean Absolute Error (MAE) :", train_mae)
print("Training Mean Squared Error (MSE)  :", train_mse)
print("Training Root Mean Squared Error (RMSE):", train_rmse)
print("Training R²:", train_r2)

print("\nTest Mean Absolute Error (MAE) :", mae)
print("Test Mean Squared Error (MSE)  :", mse)
print("Test Root Mean Squared Error (RMSE):", rmse)
print("Test R²:", r2)

## Dataset Source

Wine Quality dataset from the UCI Machine Learning Repository.

**Cortez, P., Cerdeira, A., Almeida, F., Matos, T., & Reis, J. (2009). Wine Quality.**

The dataset is used here for educational machine-learning practice. See the repository README for source and licensing information.
